# PyTorch - Transformers Experimentation

This notebook is intended to experiment the usage of Transformers in PyTorch for Time Series Forecasting.

# Notebook Setup

## Imports

In [23]:
# Import Standard Libraries
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

## Define Configurations

In [2]:
# Data path
sunspot_data_path = './../../data/raw/sunspot_data.csv'

# Read Data

In [3]:
# Read data from local path
sunspot_data = pd.read_csv(
    sunspot_data_path,
    sep=';',
    header=None,
    names=['year', 'month', 'day', 'dec_year', 'sn_value', 'sn_error', 'obs_num', 'unused1'],
    na_values=['-1'],
    index_col=False
)

# Data Preprocessing

## Sunspot Data

In [25]:
# Find the first id that has a subsequent sequence with at least 1 observation
start_id = max(sunspot_data[sunspot_data['obs_num'] == 0].index.tolist()) + 1

# Split train and test data
sunspot_data_valida = sunspot_data.iloc[start_id:].copy()
sunspot_data['sn_value'] = sunspot_data['sn_value'].astype(float)
sunspot_data_train = sunspot_data[sunspot_data['year'] < 2000]
sunspot_data_test = sunspot_data[sunspot_data['year'] >= 2000]

# Select only the column 'sn_value'
sunspot_data_train = sunspot_data_train['sn_value'].to_numpy().reshape(-1, 1)
sunspot_data_test = sunspot_data_test['sn_value'].to_numpy().reshape(-1, 1)

# Standardisation
scaler = StandardScaler()
sunspot_data_train = scaler.fit_transform(sunspot_data_train).flatten().tolist()
sunspot_data_test = scaler.transform(sunspot_data_test).flatten().tolist()

# Create different sequence batches of 10 time steps each
def to_sequences(sequence_size, observations):
    """Transform a sequence of observations into train and test sequences. (e.g., [1, 2, 3] -> [4])"""
    x, y = [], []
    for i in range(len(observations) - sequence_size):
        # Compute the current window and the subsequent element (i.e., target)
        window = observations[i:(i + sequence_size)]
        after_window = observations[i + sequence_size]

        # Append them
        x.append(window)
        y.append(after_window)

    return (torch.tensor(x, dtype=torch.float32).view(-1, sequence_size, 1),
            torch.tensor(y, dtype=torch.float32).view(-1, 1))

# to_sequence example
example_sequence = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
example_sequence_size = 4
example_sequence_output = to_sequences(example_sequence_size, example_sequence)
print('Example sequence:', example_sequence)
print('Example sequence size:', example_sequence_size)
print('Example sequence output X one element:', example_sequence_output[0][0])
print('Example sequence Output Y one element:', example_sequence_output[1][0])

# Transform train and test into sequences
sunspot_sequence_size = 10
x_train, y_train = to_sequences(sunspot_sequence_size, sunspot_data_train)
x_test, y_test = to_sequences(sunspot_sequence_size, sunspot_data_test)

# Create the Data Loader in PyTorch
train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = TensorDataset(x_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

Example sequence: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Example sequence size: 4
Example sequence output X one element: tensor([[1.],
        [2.],
        [3.],
        [4.]])
Example sequence Output Y one element: tensor([5.])
